# Movie Recommendation with Knowledge Graph Embeddings

We are going to use the best embeddings model, trained in the file `embeddings.py` and use it to implement a pseudo-recommender system that finds movies 'similar' to a given movie (based on cosine similarity).

Similar movies are going to have similar relationships: actors repeated, genres, directors... so the idea is that similar embeddings are going to also provide us similar movies.

In [1]:
import torch
import pickle
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sqlalchemy import create_engine
from urllib.parse import quote_plus

from torchkge.models import TransEModel, DistMultModel, ComplExModel

print("All libraries imported.")

All libraries imported.


C:\Users\Usuario\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torchkge\utils\data_redundancy.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [2]:
MODEL_DIR = '../saved_model'

# LOAD THE EMBEDDINGS MODEL

with open(f'{MODEL_DIR}/config.pkl', 'rb') as f:
    config = pickle.load(f)

with open(f'{MODEL_DIR}/ent_to_ix.pkl', 'rb') as f:
    ent_to_ix = pickle.load(f)
with open(f'{MODEL_DIR}/rel_to_ix.pkl', 'rb') as f:
    rel_to_ix = pickle.load(f)

ix_to_ent = {v: k for k, v in ent_to_ix.items()}
n_entities = len(ent_to_ix)
n_relations = len(rel_to_ix)

print(f"Loaded config: {config}")
print(f"Number of entities: {n_entities}")

Loaded config: {'model': 'DistMult', 'emb_dim': 100, 'lr': 0.001, 'margin': 0.5, 'n_epochs': 500, 'batch_size': 32768}
Number of entities: 17250


In [3]:
models_map = {'TransE': TransEModel, 'DistMult': DistMultModel, 'ComplEx': ComplExModel}
model_class = models_map[config['model']]

model = model_class(
    emb_dim=config['emb_dim'], 
    n_entities=n_entities, 
    n_relations=n_relations
)

model.load_state_dict(torch.load(f'{MODEL_DIR}/best_kge_model.pt'))
model.eval() 

entity_embeddings = model.ent_emb.weight.detach().cpu().numpy()

print(f"Model {config['model']} loaded and embeddings extracted.")
print(f"Embedding matrix shape: {entity_embeddings.shape}")

Model DistMult loaded and embeddings extracted.
Embedding matrix shape: (17250, 100)


C:\Users\Usuario\AppData\Local\Temp\ipykernel_16160\1383824719.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f'{MODEL_DIR}/best_kge_m

We load the original DB to get the movie title from a given movie_id (we could also use SparQL with the relation :name, but in this case using data in a tabular form is in fact better because we can obtain a simple mapping).

In [4]:
db_user = "postgres"
db_password = "postgres"
db_host = "localhost"
db_port = "5432"
db_name = "bdaproject2"
db_connection_str = f"postgresql+psycopg2://{db_user}:{quote_plus(db_password)}@{db_host}:{db_port}/{db_name}"
db_engine = create_engine(db_connection_str)
movies_df = pd.read_sql("SELECT movie_id, title FROM movies", db_engine, index_col='movie_id')

def get_title_from_uri(uri):
    try:
        movie_id = int(uri.split('/')[-1])
        return movies_df.loc[movie_id, 'title']
    except (ValueError, KeyError):
        return "(Title not found)"

print(f"Loaded {len(movies_df)} movie titles for mapping.")

Loaded 1344 movie titles for mapping.


## Simple Recommendation Engine

In [5]:
def get_recommendations(movie_title: str, top_n: int = 10):
    try:
        movie_id = movies_df[movies_df['title'].str.lower() == movie_title.lower()].index[0]
        movie_uri = f"http://www.bdaproject.org/instance/movie/{movie_id}"
        movie_idx = ent_to_ix[movie_uri]
    except (IndexError, KeyError):
        print(f"Error: Movie '{movie_title}' not found in the knowledge graph.")
        return
    # Embeddings of entitites
    target_vector = entity_embeddings[movie_idx].reshape(1, -1)
    similarities = cosine_similarity(target_vector, entity_embeddings)[0]

    # Most similar:
    similar_indices = np.argsort(similarities)[::-1]

    recommendations = []
    print(f"--- Recommendations for '{movie_title}' ---")
    for idx in similar_indices:
        if idx == movie_idx: # Skip itself :)
            continue
        
        entity_uri = ix_to_ent[idx]
        if "/movie/" in entity_uri:
            title = get_title_from_uri(entity_uri)
            similarity_score = similarities[idx]
            recommendations.append((title, similarity_score))
            print(f"- {title} (Similarity: {similarity_score:.4f})")
            
        if len(recommendations) >= top_n:
            break
    
    return recommendations

## Examples

In [6]:
_ = get_recommendations("The Dark Knight")

--- Recommendations for 'The Dark Knight' ---
- Good Night, and Good Luck. (Similarity: 0.8772)
- Batman Returns (Similarity: 0.8762)
- From Dusk Till Dawn (Similarity: 0.8760)
- Batman Forever (Similarity: 0.8733)
- The Holiday (Similarity: 0.8709)
- Batman Begins (Similarity: 0.8664)
- Constantine (Similarity: 0.8656)
- Confessions of a Dangerous Mind (Similarity: 0.8652)
- Vanilla Sky (Similarity: 0.8617)
- Pulp Fiction (Similarity: 0.8612)


The Dark Knight is a batman movie. We see that the other movies from the Batman Series also appear, together with noir-style thrillers, psychological dramas...

Even though we don't use information about the description of the movie (we could have included it using NLP embeddings), it seems that the recommendations do make some sense and are in general "serious", "action" and "dark" films.

In [7]:
_ = get_recommendations("Spirited Away", top_n=5)

--- Recommendations for 'Spirited Away' ---
- Howl's Moving Castle (Similarity: 0.8965)
- Princess Mononoke (Similarity: 0.8666)
- Toy Story (Similarity: 0.8434)
- (Title not found) (Similarity: 0.8426)
- Paris Je T'aime (Similarity: 0.8274)


In this case, the original movie was Studio Ghibli's Spirited Away. We get recommendations with are very strongly related: Howl's Moving Castle, Princes Mononoke... (same production company, same director). We also get Toy Story, which is also oriented to kids.

In [8]:
_ = get_recommendations("The Godfather")

--- Recommendations for 'The Godfather' ---
- The Godfather Part III (Similarity: 0.8120)
- Meet the Parents (Similarity: 0.8096)
- Signs (Similarity: 0.8096)
- The Darjeeling Limited (Similarity: 0.8071)
- The Good Shepherd (Similarity: 0.8045)
- Pirates of the Caribbean: On Stranger Tides (Similarity: 0.8043)
- Indiana Jones and the Last Crusade (Similarity: 0.8023)
- Don Juan DeMarco (Similarity: 0.7994)
- Entrapment (Similarity: 0.7991)
- The Last Samurai (Similarity: 0.7978)


Finally, for The GodFather we see that the top recommendation is one of its sequels. We also have The Good Shepherd (directed by Robert De Niro), Don Juan DeMarco (which features Marlon Brando)...

However, some of the recommendations do not make a lot of sense: Signs, The Darjeeling Limited, Pirates of the Caribbean, Indiana Jones... Those are movies which don't have a lot to do with The Godfather. Ideally, we could combine this embeddings with others (like the NLP option) or distance based metrics to further enhance the recommender. Also, if we had more information about users and reviews, we could use some collaborative-filtering approaches to combine with this content-based idea.